# Multi-query retriever

Muchas veces una persona hace una pregunta y quien recibe esa pregunta no la entiende. Con lo cual la persona vuelve a formular la pregunta de otra manera para que la otra persona pueda entenderla mejor. Varias veces se repite este proceso hasta que la persona que recibe la pregunta entiende lo que se le está preguntando.

Lo mismo ocurre con los modelos de lenguaje y los sistemas de recuperación de información. Muchas veces la pregunta que se hace no es lo suficientemente clara para que el sistema de recuperación pueda encontrar la información relevante. Aqui entra en juego el **multi-query retriever**. Este componente toma una pregunta y genera múltiples reformulaciones de la misma. Luego, utiliza estas reformulaciones para buscar en la base de datos y recuperar la información más relevante.

![Diagrama](../docs/media/diagrams/03.png)

## Librerías

In [39]:
import logging
import re
from typing import List

from dotenv import load_dotenv
from langchain_classic.chains import LLMChain
from langchain_mistralai import ChatMistralAI
from langchain_mistralai import MistralAIEmbeddings
from langchain_core.output_parsers import BaseOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_classic.retrievers import MultiQueryRetriever
from langchain_chroma import Chroma
from pydantic import BaseModel, Field

from src.langchain_docs_loader import load_langchain_docs_splitted

load_dotenv()

True

## Carga de datos

In [4]:
docs = load_langchain_docs_splitted()
len(docs)

555

## Preparación de vectorstore

In [5]:
embedding = MistralAIEmbeddings()
vectorstore = Chroma.from_documents(documents=docs, embedding=embedding)

## Preparación de retriever

In [6]:
llm = ChatMistralAI(temperature=0)
retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(),
    llm=llm,
)

In [7]:
logging.basicConfig()
logging.getLogger("langchain_classic.retrievers.multi_query").setLevel(logging.INFO)

##  Prueba de retriever

In [9]:
retriever.invoke(
    "How to create a retriever with langchain expression language?"
)

INFO:langchain_classic.retrievers.multi_query:Generated queries: ['Here are three alternative versions of your question to help retrieve relevant documents from a vector database:', '1. "What is the process for building a retriever using LangChain\'s expression language?"', '2. "How can I implement a retriever in LangChain with expression language?"', '3. "Steps to construct a retriever with LangChain\'s expression language syntax"']


[Document(id='d63467cf-3350-4040-9165-f7b3fcb276e2', metadata={'source': 'https://docs.langchain.com/oss/python/langchain/knowledge-base', 'title': 'Build a semantic search engine with LangChain - Docs by LangChain', 'language': 'en'}, page_content='```\n\nCopy\n``` \n[[Document(metadata={\'page\': 4, \'source\': \'../example_data/nke-10k-2023.pdf\', \'start_index\': 3125}, page_content=\'direct to consumer operations sell products through the following number of retail stores in the United States:\\nU.S. RETAIL STORES NUMBER\\nNIKE Brand factory stores 213 \\nNIKE Brand in-line stores (including employee-only stores) 74 \\nConverse stores (including factory stores) 82 \\nTOTAL 369 \\nIn the United States, NIKE has eight significant distribution centers. Refer to Item 2. Properties for further information.\\n2023 FORM 10-K 2\')],\n [Document(metadata={\'page\': 3, \'source\': \'../example_data/nke-10k-2023.pdf\', \'start_index\': 0}, page_content=\'Table of Contents\\nPART I\\nITEM 1. 

## Generación de preguntas alternativas de forma personalizada

### Definición de esquema de salida de preguntas

In [48]:
class LineListOutputParser(BaseOutputParser[List[str]]):
    def parse(self, text: str) -> List[str]:
        lines = text.strip().split("\n")
        cleaned_lines = []
        pattern = r"^\d+\." # Patrón para detectar números al inicio de la línea
        
        for line in lines:
            line = line.strip()

            # SOLO procesamos la línea si coincide con el patrón "Numero + Punto"
            if re.match(pattern, line):
                # Limpiamos comillas extra si las hay
                line_content = line.strip('"').strip("'")

                cleaned_lines.append(line_content)

        return cleaned_lines

### Creación de `prompt` personalizado

In [49]:
prompt = PromptTemplate.from_template(
    """You are an AI language assistant well versed in the Langchain Documentation.
Your more precise task is to generate five different versions of the given question to retrieve relevant documents from a vector database.
By generating multiple perspectives on the question, your goal is to overcome some of the limitations of the distance-based similarity search.

Provide these alternative questions separed by newlines.

Original question: {question}
New questions:"""
)

# Usando Langchain Classic
# llm_chain = LLMChain(
#     llm=ChatMistralAI(temperature=0),
#     prompt=prompt,
#     output_parser=LineListOutputParser(),
# )

# Usando LCEL podemos encadenar los componentes con el operador pipe |
llm_chain = prompt | llm | LineListOutputParser()

### Use de cadena de generación de preguntas personalizada

In [50]:
llm_chain.invoke(
    {"question": "How to create a retriever with langchain expression language?"}
)

['1. "What is the process for building a retriever using LangChain\'s expression language?',
 '2. "How can I implement a retriever in LangChain with expression language syntax?',
 '3. "Steps to construct a retriever with LangChain\'s expression language features',
 '4. "Guide to creating a retriever using LangChain\'s expression language capabilities',
 '5. "How does LangChain\'s expression language help in developing a retriever?']

### Integración de cadena de generación de preguntas personalizada en `retriever`

In [51]:
retriever = MultiQueryRetriever(
    retriever=vectorstore.as_retriever(),
    llm_chain=llm_chain,
)

### Uso de `retriever` con cadena de generación de preguntas personalizada

In [52]:
retriever.invoke(
    "How to create a retriever with lagnchain expression language?"
)

INFO:langchain_classic.retrievers.multi_query:Generated queries: ['1. "What is the process for building a retriever using LangChain\'s expression language?', '2. "How can I implement a retriever in LangChain with expression language?', '3. "Steps to construct a retriever with LangChain\'s expression language syntax', '4. "Guide to creating a retriever using LangChain\'s expression language features', '5. "How to use LangChain\'s expression language to define and configure a retriever']


[Document(id='2528cb29-49f4-4d70-9362-835c25fef06f', metadata={'language': 'en', 'source': 'https://docs.langchain.com/oss/python/langchain/knowledge-base', 'title': 'Build a semantic search engine with LangChain - Docs by LangChain'}, page_content='## \u200b4. Retrievers\n\nLangChain [VectorStore](https://reference.langchain.com/python/langchain_core/vectorstores/?h=#langchain_core.vectorstores.base.VectorStore) objects do not subclass [Runnable](https://reference.langchain.com/python/langchain_core/runnables/#langchain_core.runnables.Runnable). LangChain [Retrievers](https://reference.langchain.com/python/langchain_core/retrievers/#langchain_core.retrievers.BaseRetriever) are Runnables, so they implement a standard set of methods (e.g., synchronous and asynchronous invoke and batch operations). Although we can construct retrievers from vector stores, retrievers can interface with non-vector store sources of data, as well (such as external APIs).\n\nWe can create a simple version of t